# NYT Factor Pipeline — Small Sample Demo

This notebook demonstrates the end-to-end pipeline using synthetic data.
No API keys are needed to run this demo.

## What this demo shows
1. Initialize the database
2. Insert synthetic articles
3. Score and filter articles
4. Embed articles (using local model)
5. Cluster and discover themes
6. Build theme time series
7. Ingest sample companies
8. Score companies against themes

In [ ]:
import sys
import json
import numpy as np
from datetime import datetime, date, timedelta
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

from nyt_factor_pipeline.db import init_db
from nyt_factor_pipeline.config import get_settings, reset_settings
from nyt_factor_pipeline.logging_utils import setup_logging

setup_logging()

# Use in-memory DB for demo
import duckdb
from nyt_factor_pipeline.db import init_schema
conn = duckdb.connect(':memory:')
init_schema(conn)
print('Database initialized!')

In [ ]:
# Step 1: Insert synthetic articles
from nyt_factor_pipeline.ingest.normalize import normalize_article

# Generate synthetic articles across 3 distinct topics
topics = {
    'ai_tech': {
        'headlines': [
            'AI Revolution Transforms Silicon Valley',
            'New AI Chips Break Performance Records',
            'Tech Giants Race to Build AI Infrastructure',
            'AI Startups Raise Record Venture Capital',
            'Artificial Intelligence Reshapes Software Industry',
            'GPU Shortage Drives Up AI Computing Costs',
            'Major AI Model Achieves Human-Level Reasoning',
            'Cloud Computing Firms Invest Billions in AI',
        ],
        'section': 'Technology',
        'desk': 'Technology',
        'keywords': [{'value': 'Artificial Intelligence'}, {'value': 'Technology'}, {'value': 'Semiconductors'}],
    },
    'fed_rates': {
        'headlines': [
            'Federal Reserve Holds Interest Rates Steady',
            'Inflation Data Sparks Rate Cut Speculation',
            'Bond Yields Fall on Fed Policy Outlook',
            'Central Banks Signal Cautious Easing Path',
            'Housing Market Reacts to Rate Decision',
            'Dollar Weakens as Rate Expectations Shift',
            'Treasury Market Volatility Increases',
            'Fed Officials Debate Timing of Rate Cuts',
        ],
        'section': 'Business Day',
        'desk': 'Business/Financial Desk',
        'keywords': [{'value': 'Federal Reserve'}, {'value': 'Interest Rates'}, {'value': 'Monetary Policy'}],
    },
    'energy_transition': {
        'headlines': [
            'Solar Energy Installations Hit Record High',
            'Oil Prices Surge on Middle East Tensions',
            'Electric Vehicle Sales Surpass Expectations',
            'New Battery Technology Promises Longer Range',
            'Wind Farm Projects Gain Government Approval',
            'Energy Companies Accelerate Green Investments',
            'Carbon Capture Technology Attracts Major Funding',
            'Natural Gas Prices Spike During Cold Snap',
        ],
        'section': 'Business Day',
        'desk': 'Business',
        'keywords': [{'value': 'Energy'}, {'value': 'Climate Change'}, {'value': 'Oil'}],
    },
}

articles_inserted = 0
for topic_name, topic_data in topics.items():
    for i, headline in enumerate(topic_data['headlines']):
        day_offset = i * 2
        raw = {
            'uri': f'nyt://article/{topic_name}-{i}',
            'web_url': f'https://nytimes.com/2024/01/{10+day_offset:02d}/{topic_name}.html',
            'headline': {'main': headline},
            'abstract': f'{headline}. Analysis of recent developments.',
            'snippet': f'Coverage of {headline.lower()}.',
            'lead_paragraph': f'{headline}. Experts say this development has significant implications for the economy and markets.',
            'pub_date': f'2024-01-{10+day_offset:02d}T10:00:00+0000',
            'source': 'The New York Times',
            'section_name': topic_data['section'],
            'news_desk': topic_data['desk'],
            'type_of_material': 'News',
            'document_type': 'article',
            'print_section': 'A',
            'print_page': str(1 + i % 3),
            'word_count': 800 + i * 100,
            'byline': {'original': 'By Test Reporter'},
            'keywords': topic_data['keywords'],
            'multimedia': [],
        }
        article = normalize_article(raw, 'synthetic')
        if article:
            conn.execute(
                '''INSERT INTO articles (article_id, source_api, web_url, uri, pub_date, year, month, day,
                    headline_main, abstract, snippet, lead_paragraph, source, section_name, subsection_name,
                    news_desk, type_of_material, document_type, print_section, print_page, word_count,
                    byline_original, keywords_json, multimedia_json, normalized_text, importance_score, macro_relevance_score)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                ON CONFLICT (article_id) DO NOTHING''',
                [article.article_id, article.source_api, article.web_url, article.uri,
                 article.pub_date, article.year, article.month, article.day,
                 article.headline_main, article.abstract, article.snippet, article.lead_paragraph,
                 article.source, article.section_name, article.subsection_name, article.news_desk,
                 article.type_of_material, article.document_type, article.print_section, article.print_page,
                 article.word_count, article.byline_original, article.keywords_json, article.multimedia_json,
                 article.normalized_text, article.importance_score, article.macro_relevance_score]
            )
            articles_inserted += 1

print(f'Inserted {articles_inserted} synthetic articles')
conn.execute('SELECT section_name, COUNT(*) FROM articles GROUP BY section_name').fetchall()

In [ ]:
# Step 2: Score articles
from nyt_factor_pipeline.scoring.article_importance import score_articles_in_db
from nyt_factor_pipeline.scoring.article_filtering import compute_macro_relevance

scored = score_articles_in_db(conn)
relevant = compute_macro_relevance(conn)
print(f'Scored {scored} articles')
print(f'Macro-relevant: {relevant}')

# Show score distribution
scores = conn.execute(
    'SELECT headline_main, importance_score, macro_relevance_score FROM articles ORDER BY importance_score DESC LIMIT 10'
).fetchall()
for h, s, mr in scores:
    print(f'  {s:.3f} (rel={mr:.0f}) {h[:60]}')

In [ ]:
# Step 3: Embed articles
# NOTE: This requires sentence-transformers. If not installed, use random embeddings.
from nyt_factor_pipeline.scoring.article_filtering import get_embeddable_articles

articles = get_embeddable_articles(conn, min_importance=0.0, min_word_count=0)
print(f'Articles to embed: {len(articles)}')

try:
    from nyt_factor_pipeline.embeddings.embedder import embed_articles
    embedded = embed_articles(conn, articles, batch_size=32)
    print(f'Embedded {embedded} articles using sentence-transformers')
except ImportError:
    # Fallback: use random embeddings for demo
    import pickle
    print('sentence-transformers not available, using random embeddings for demo')
    for a in articles:
        emb = np.random.randn(384).astype(np.float32)
        emb = emb / np.linalg.norm(emb)
        conn.execute(
            '''INSERT INTO article_embeddings (article_id, embedding, embedding_model)
               VALUES (?, ?, 'random-demo')''',
            [a['article_id'], pickle.dumps(emb)]
        )
    print(f'Embedded {len(articles)} articles with random vectors')

In [ ]:
# Step 4: Cluster articles
from nyt_factor_pipeline.clustering.cluster_weekly import cluster_date_range

clusters = cluster_date_range(
    conn,
    start_date=date(2024, 1, 1),
    end_date=date(2024, 1, 31),
    window_type='weekly',
)
print(f'Found {len(clusters)} clusters')
for c in clusters:
    print(f'  {c["cluster_id"]}: {c["article_count"]} articles, keywords: {c["top_keywords"][:5]}')

In [ ]:
# Step 5: Track themes
from nyt_factor_pipeline.clustering.topic_tracking import track_themes

stats = track_themes(conn)
print(f'Theme tracking: {stats}')

themes = conn.execute('SELECT theme_id, current_label, first_seen, last_seen FROM themes').fetchall()
for t in themes:
    print(f'  {t[0]}: "{t[1]}" ({t[2]} to {t[3]})')

In [ ]:
# Step 6: Build timeseries
from nyt_factor_pipeline.themes.timeseries import build_theme_timeseries, get_theme_timeseries
from nyt_factor_pipeline.themes.burst_detection import compute_burst_zscores

ts_count = build_theme_timeseries(conn)
burst_count = compute_burst_zscores(conn)
print(f'Built {ts_count} timeseries rows, {burst_count} burst z-scores')

# Show timeseries for first theme
if themes:
    ts = get_theme_timeseries(conn, themes[0][0])
    print(f'\nTimeseries for "{themes[0][1]}":')
    for row in ts[:5]:
        print(f'  {row["date"]}: count={row["article_count"]}, intensity={row["intensity"]:.4f}')

In [ ]:
# Step 7: Ingest sample companies and score
from nyt_factor_pipeline.exposures.company_ingest import ingest_companies_csv

csv_path = Path('.').resolve().parent / 'data' / 'sample_companies.csv'
if csv_path.exists():
    count = ingest_companies_csv(conn, csv_path)
    print(f'Ingested {count} companies')

    companies = conn.execute('SELECT ticker, company_name, rbics_name FROM companies LIMIT 10').fetchall()
    for c in companies:
        print(f'  {c[0]}: {c[1]} ({c[2]})')
else:
    print(f'Sample CSV not found at {csv_path}')

In [ ]:
# Step 8: Summary
print('=== Pipeline Summary ===')
print(f'Articles:    {conn.execute("SELECT COUNT(*) FROM articles").fetchone()[0]}')
print(f'Embedded:    {conn.execute("SELECT COUNT(*) FROM article_embeddings").fetchone()[0]}')
print(f'Clusters:    {conn.execute("SELECT COUNT(*) FROM clusters_raw").fetchone()[0]}')
print(f'Themes:      {conn.execute("SELECT COUNT(*) FROM themes").fetchone()[0]}')
print(f'Timeseries:  {conn.execute("SELECT COUNT(*) FROM theme_timeseries").fetchone()[0]}')
print(f'Companies:   {conn.execute("SELECT COUNT(*) FROM companies").fetchone()[0]}')
print()
print('Note: LLM labeling, RBICS mapping, and company scoring')
print('require OPENAI_API_KEY and are demonstrated in the CLI.')